# Attitude + Conical Sensor — 3D RTN Workflow

Extends `workflow_3d_rtn.ipynb` with rigid-body attitude dynamics and a
body-frame conical sensor — the 3D counterpart to
`workflow_2d_rt_attitude.ipynb`.

The key differences from the 2D version:
- **Frame**: `DynamicsKey.HCW_RTN`, `Frame.RTN` — full 3D relative motion.
- **Attitude**: full SO(3) rotation; `omega_max=(0.5, 0.5, 0.5)` saturates
  all three axes equally.  The torque command field is 3-dimensional.
- **Trajectory plots**: four projections (R-T, R-N, T-N, 3D RTN).
- **Animation**: 3D axes; `draw_cone_3d` draws the cone(s) as a 3D surface
  rendered by `RolloutScene`.

**Setup**:
```bash
JAX_PLATFORMS=cpu uv run --with jupyter --with ipykernel jupyter notebook examples/workflow_3d_rtn_attitude.ipynb
```


## 0. Device setup (Apple Silicon safety guard)

`orbital_game` defaults to CPU + float32 at import so it works on every JAX backend (CPU, CUDA, MPS). For full orbit precision call `orbital_game.set_precision(jnp.float64)` after import — this enables `jax_enable_x64`, which Apple's MLX backend rejects, so if `jax-mps` is installed we also pin `JAX_PLATFORMS=cpu` here as a defensive layer before importing `jax`. See `examples/running_on_mps.ipynb` for the full story.

In [1]:
import importlib.metadata
import os

try:
    importlib.metadata.distribution('jax-mps')
    os.environ.setdefault('JAX_PLATFORMS', 'cpu,mps')
    os.environ.setdefault('JAX_DEFAULT_DEVICE', 'cpu')
except importlib.metadata.PackageNotFoundError:
    os.environ.setdefault('JAX_PLATFORMS', 'cpu')

In [2]:
%load_ext autoreload
%autoreload 2

import pathlib
import shutil
import tempfile
from types import SimpleNamespace

import jax
import jax.numpy as jnp
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.animation as mpl_animation

from orbital_game import (
    OrbitalGameEnv,
    ScenarioConfig,
    VehicleParamsSpec,
    rollout,
    save_run,
    load_run,
)
from orbital_game.belief import EKFBeliefUpdater, EKFFromTruthInitializer
from orbital_game.dynamics.attitude import AttitudeParams
from orbital_game.dynamics.hcw import hcw_rtn_step
from orbital_game.env.types import Actions, BySide, Side
from orbital_game.observations.conical import ConicalObservation
from orbital_game.policies import ZeroControl
from orbital_game.reference_orbit import ReferenceOrbitState, mean_motion as ref_mean_motion
from orbital_game.registry import (
    ActionComponentKey,
    AttitudeDynamicsKey,
    DynamicsKey,
    Frame,
    StateComponentKey,
)
from orbital_game.sampling.attitude import FixedAttitude
from orbital_game.sampling.mass import ConstantMass
from orbital_game.sampling.side import RelativeEllipse
from orbital_game.sampling.spec import ICSpec
from orbital_game.viz import (
    RolloutScene,
    interactive_viewer,
    plot_reward_curve,
    plot_rollout_panels,
    save_animation,
)
from orbital_game.viz import plot_rt, plot_rn, plot_tn, plot_rtn_3d

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Platform 'mps' is experimental and not all JAX functionality may be correctly supported!


## 1. Scenario configuration — attitude + conical sensor (3D)

The 3D scenario differs from the 2D in three ways:

1. **Dynamics frame** — `HCW_RTN` / `Frame.RTN` replaces `HCW_RT` / `Frame.RT`.
   The spatial state is 6-dimensional (`[R, T, N, Ṙ, Ṫ, Ṅ]`); the EKF
   propagates a 6×6 covariance.

2. **Torque dimensionality** — `AttitudeControl.rotation_dim=3` (full SO(3));
   the command field `torque` is a 3-vector.  In 2D it was 1-D (body-z only).

3. **omega_max** — all three axes saturate at 0.5 rad/s, enabling full 3D
   attitude maneuvers.

The four-face boresight configuration is the same; in 3D the cones can point
in any direction as the spacecraft tumbles through SO(3).


### AttitudeSampler family

The new `AttitudeSampler` family (`IdentityAttitude`, `FixedAttitude`, `UniformAttitude`, `UniformBodyRates`, `UniformAttitudeAndRates`) plugs into `RelativeKeplerian` and `RelativeEllipse` via their `attitude_sampler` field. We use `FixedAttitude` here for a deterministic slow spin (0.1°/s about all three axes) so the cones sweep visibly across the rollout. Switch to `UniformBodyRates(omega_max_rad_s=(...))` for randomized initial spin or `UniformAttitude()` for randomized initial orientation.

In [3]:
sensor_boresights_body = jnp.array([
    [ 1.0,  0.0, 0.0],   # +R face
    [-1.0,  0.0, 0.0],   # -R face
    [ 0.0,  1.0, 0.0],   # +T face
    [ 0.0, -1.0, 0.0],   # -T face
], dtype=jnp.float32)

# Alternative: single +R face only.
# sensor_boresights_body = jnp.array([[1.0, 0.0, 0.0]], dtype=jnp.float32)

# Alternative: two opposing faces.
# sensor_boresights_body = jnp.array([
#     [1.0, 0.0, 0.0],
#     [-1.0, 0.0, 0.0],
# ], dtype=jnp.float32)

half_angle_rad = jnp.deg2rad(15.0)   # ±15 deg half-angle per cone face (narrow = intermittent)

# Full SO(3) attitude params: equal inertia, all-axis saturation at 0.5 rad/s.
attitude_params = AttitudeParams(
    inertia_diag=jnp.array([10.0, 10.0, 10.0], dtype=jnp.float32),
    omega_max=jnp.array([0.5, 0.5, 0.5], dtype=jnp.float32),
)

# ── Attitude IC sampler: slow deterministic tumble via FixedAttitude ───────
# 0.1 deg/s on all three axes × 10 s/step × 200 steps = 200 deg total per axis.
# Replace with UniformBodyRates or UniformAttitude for randomized ICs.
_slow_spin_3d = FixedAttitude(
    quat_wxyz=(1.0, 0.0, 0.0, 0.0),
    omega_rad_s=(
        float(jnp.deg2rad(0.1)),
        float(jnp.deg2rad(0.1)),
        float(jnp.deg2rad(0.1)),
    ),
)

# Proto config to get the layout.
proto_cfg = ScenarioConfig(
    n_guards=1,
    n_bandits=1,
    epoch_mjd_utc=60067.0,
    reference_orbit=ReferenceOrbitState(
        position_eci=jnp.array([7000e3, 0.0, 0.0]),
        velocity_eci=jnp.array([0.0, 7.5e3, 0.0]),
    ),
    guard_components=(
        StateComponentKey.RTN,
        StateComponentKey.MASS,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    bandit_components=(
        StateComponentKey.RTN,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    guard_params=VehicleParamsSpec(dry_mass_kg=100.0, isp_s=220.0, max_thrust_n=5.0),
    bandit_params=VehicleParamsSpec(dry_mass_kg=50.0, isp_s=200.0, max_thrust_n=2.0),
    ic_sampler=ICSpec(
        guard_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=0.0, sigma_radial_ellipse_m=10.0,
            mass_sampler=ConstantMass(propellant_mass_kg=10.0),
            attitude_sampler=_slow_spin_3d,
        ),
        bandit_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=jnp.pi, sigma_radial_ellipse_m=10.0,
            attitude_sampler=_slow_spin_3d,
        ),
    ),
    truth_dynamics=DynamicsKey.HCW_RTN,
    policy_dynamics=DynamicsKey.HCW_RTN,
    action_frame=Frame.RTN,
    guard_action_components=(
        ActionComponentKey.ATTITUDE_CONTROL,
        ActionComponentKey.IMPULSIVE_MANEUVER,
    ),
    bandit_action_components=(ActionComponentKey.IMPULSIVE_MANEUVER,),
    attitude_dynamics_key=AttitudeDynamicsKey.RIGID_BODY,
    guard_attitude_params=attitude_params,
    bandit_attitude_params=attitude_params,
    dt=10.0,
    max_horizon_s=2000.0,
    seed=0,
)
proto_layout = OrbitalGameEnv(proto_cfg).layout

guard_obs_fn = ConicalObservation(
    layout=proto_layout,
    sensor_boresights_body=sensor_boresights_body,
    half_angle_rad=float(half_angle_rad),
    sigma=2.0,
)

# Symmetric sensor configuration on both sides — the bandit also has a
# body-fixed cone fan and must rotate to track the guard. This makes both
# sides' beliefs visibly fluctuate as their cones sweep.
bandit_obs_fn = ConicalObservation(
    layout=proto_layout,
    sensor_boresights_body=sensor_boresights_body,
    half_angle_rad=float(half_angle_rad),
    sigma=2.0,
)

cfg = ScenarioConfig(
    n_guards=1,
    n_bandits=1,
    epoch_mjd_utc=60067.0,
    reference_orbit=ReferenceOrbitState(
        position_eci=jnp.array([7000e3, 0.0, 0.0]),
        velocity_eci=jnp.array([0.0, 7.5e3, 0.0]),
    ),
    guard_components=(
        StateComponentKey.RTN,
        StateComponentKey.MASS,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    bandit_components=(
        StateComponentKey.RTN,
        StateComponentKey.ATTITUDE,
        StateComponentKey.BODY_RATES,
    ),
    guard_params=VehicleParamsSpec(dry_mass_kg=100.0, isp_s=220.0, max_thrust_n=5.0),
    bandit_params=VehicleParamsSpec(dry_mass_kg=50.0, isp_s=200.0, max_thrust_n=2.0),
    ic_sampler=ICSpec(
        guard_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=0.0, sigma_radial_ellipse_m=10.0,
            mass_sampler=ConstantMass(propellant_mass_kg=10.0),
            attitude_sampler=_slow_spin_3d,
        ),
        bandit_sampler=RelativeEllipse(
            radial_ellipse_m=1000.0, phase_rad=jnp.pi, sigma_radial_ellipse_m=10.0,
            attitude_sampler=_slow_spin_3d,
        ),
    ),
    truth_dynamics=DynamicsKey.HCW_RTN,
    policy_dynamics=DynamicsKey.HCW_RTN,
    action_frame=Frame.RTN,
    guard_action_components=(
        ActionComponentKey.ATTITUDE_CONTROL,
        ActionComponentKey.IMPULSIVE_MANEUVER,
    ),
    bandit_action_components=(ActionComponentKey.IMPULSIVE_MANEUVER,),
    attitude_dynamics_key=AttitudeDynamicsKey.RIGID_BODY,
    guard_attitude_params=attitude_params,
    bandit_attitude_params=attitude_params,
    guard_observation_fn=guard_obs_fn,
    bandit_observation_fn=bandit_obs_fn,
    dt=10.0,
    max_horizon_s=2000.0,
    seed=0,
)

print(f"max_steps (derived):     {cfg.max_steps}")
print(f"guard action components: {[c.value for c in cfg.guard_action_components]}")
print(f"guard obs fn:            {type(cfg.guard_observation_fn).__name__}")
print(f"bandit obs fn:           {type(cfg.bandit_observation_fn).__name__}")
print(f"half_angle_deg:          {float(jnp.rad2deg(half_angle_rad)):.1f}")
print(f"IC spin rate (deg/s):    0.1 per axis")
print(f"  → 0.1 deg/s × 10s × 200 steps = {0.1*10*200:.0f} deg total per axis")

max_steps (derived):     200
guard action components: ['attitude_control', 'impulsive_maneuver']
guard obs fn:            ConicalObservation
bandit obs fn:           ConicalObservation
half_angle_deg:          15.0
IC spin rate (deg/s):    0.1 per axis
  → 0.1 deg/s × 10s × 200 steps = 200 deg total per axis


## 2. Construct the environment

The flat state dim is larger than in the plain 3D RTN notebook.
Guard: RTN(6) + MASS(1) + ATTITUDE(4) + BODY_RATES(3) + APPLIED_TORQUE(3) + APPLIED_DV(3) = 20.
Bandit: RTN(6) + ATTITUDE(4) + BODY_RATES(3) + APPLIED_TORQUE(3) + APPLIED_DV(3) = 19.


In [4]:
env = OrbitalGameEnv(cfg)
print(f"mean motion (rad/s):  {env.mean_motion:.3e}")
print(f"layout flat dim:      {env.layout.flat_dim}")
print(f"guard command fields: {list(env.guard_command_cls.zeros(1).__dataclass_fields__.keys())}")


mean motion (rad/s):  1.098e-03
layout flat dim:      39
guard command fields: ['torque', 'dv']


## 3. Policies — passive rotation only

Both sides use `ZeroControl`: no Δv, no torque commands.  Attitude evolves
*passively* from the body rate set by the IC sampler (0.1 °/s about each
body axis in 3D RTN), so the cones sweep slowly and smoothly across the
rollout.  Free precession with zero torque means the joint
quaternion-and-omega ODE collapses to pure RK4 on
\(\dot{q}=\tfrac{1}{2}\,\Omega(\omega)\,q\), with the gyroscopic
term keeping \(\omega\) constant for a symmetric body.

To turn this into an *active* pointing problem, the user would write a
heuristic that reads the bandit position from `env_state` (or the guard's
belief), compute the angle error in 3D between the +x boresight and the
line-of-sight, and command three-axis torque proportional to that error.
We keep the demonstration controller-free here so the cone-and-belief
story is visually unambiguous.

In [5]:
# Both sides emit zero commands; attitude rotates passively from the IC.
guard_policy = ZeroControl(n_vehicles=cfg.n_guards, command_cls=env.guard_command_cls)
bandit_policy = ZeroControl(n_vehicles=cfg.n_bandits, command_cls=env.bandit_command_cls)

def init_ps(config, env_state, key):
    del config, env_state, key
    return None

print("Guard policy:  ZeroControl (passive rotation from IC body rate)")
print("Bandit policy: ZeroControl (passive rotation from IC body rate)")


Guard policy:  ZeroControl (passive rotation from IC body rate)
Bandit policy: ZeroControl (passive rotation from IC body rate)


## 4. Manual rollout with attitude state (3D)

Same hand-rolled loop as the 2D notebook, adapted for 3D HCW dynamics in
the EKF.  `per_vehicle_hcw_rtn` propagates the 6-dim state; process noise
is 6×6.


In [6]:
from orbital_game.env.types import SideTrajectory, Trajectory
from orbital_game.observations.types import flatten_observations

N_STEPS = cfg.max_steps

n_motion = float(ref_mean_motion(cfg.reference_orbit))
hcw_params = SimpleNamespace(mean_motion=n_motion)
d = env.layout.dynamics_state_dim  # 6 for RTN

def per_vehicle_hcw_rtn(x, u, dt):
    return hcw_rtn_step(x[None, :], u[None, :], hcw_params, dt)[0]

ekf_updater = EKFBeliefUpdater(
    dynamics_fn=per_vehicle_hcw_rtn,
    process_noise=jnp.eye(d, dtype=jnp.float32) * 0.01,
    dt=cfg.dt,
)
ekf_initializer = EKFFromTruthInitializer(
    layout=env.layout,
    variance_diag=jnp.ones(d, dtype=jnp.float32) * 100.0,
)

# ── Manual rollout loop ────────────────────────────────────────────────────
# IC body rates come from FixedAttitude via the attitude_sampler field —
# no manual state.replace() injection needed.
key = jax.random.PRNGKey(42)
k_reset, key = jax.random.split(key)
state, initial_outputs = env.reset(k_reset)


k_init, key = jax.random.split(key)
belief_guard = ekf_initializer(state, Side.GUARD, k_init)
k_init_b, key = jax.random.split(key)
belief_bandit = ekf_initializer(state, Side.BANDIT, k_init_b)

zero_dv = jnp.zeros((cfg.n_guards, 3), dtype=jnp.float32)
zero_guard_cmd = env.guard_command_cls.zeros(cfg.n_guards)
zero_bandit_cmd = env.bandit_command_cls.zeros(cfg.n_bandits)
identity_actions = Actions(sides=BySide(guard=zero_guard_cmd, bandit=zero_bandit_cmd))

state_history          = []
obs_history            = []
reward_history         = []
guard_action_history   = []
bandit_action_history  = []
episode_done_history   = []
guard_obs_flat_history = []
bandit_obs_flat_history= []
belief_guard_history   = []
belief_bandit_history  = []

for step in range(N_STEPS):
    k_guard, k_bandit, k_env, k_obs, k_upd, key = jax.random.split(key, 6)

    guard_cmd, _ = guard_policy(None, None, k_guard, state.t)
    bandit_cmd = zero_bandit_cmd
    actions = Actions(sides=BySide(guard=guard_cmd, bandit=bandit_cmd))

    step_out = env.step(k_env, state, actions)
    state = step_out.state
    
    obs_channels = guard_obs_fn(state, actions, Side.GUARD, cfg, k_obs, state.t)
    obs = obs_channels[0]

    # Bandit conical observation on the new state.
    bandit_obs_channels = bandit_obs_fn(state, actions, Side.BANDIT, cfg, k_upd, state.t)

    # EKF update for both sides (predict + correct where visible).
    k_upd_g, k_upd_b = jax.random.split(k_upd)
    belief_guard = ekf_updater(belief_guard, obs_channels, zero_dv, Side.GUARD, k_upd_g)
    zero_dv_b = jnp.zeros((cfg.n_bandits, 3), dtype=jnp.float32)
    belief_bandit = ekf_updater(belief_bandit, bandit_obs_channels, zero_dv_b, Side.BANDIT, k_upd_b)

    state_history.append(state)
    obs_history.append(obs)
    reward_history.append(step_out.outputs.guard.reward)
    guard_action_history.append(guard_cmd)
    bandit_action_history.append(bandit_cmd)
    episode_done_history.append(step_out.episode_done)
    guard_obs_flat_history.append(flatten_observations(step_out.outputs.guard.obs))
    bandit_obs_flat_history.append(flatten_observations(step_out.outputs.bandit.obs))
    belief_guard_history.append(belief_guard)
    belief_bandit_history.append(belief_bandit)

rewards_arr = jnp.stack(reward_history, axis=0)
print(f"Episode return: {float(rewards_arr.sum()):.2f}")
print(f"Steps with at least one in-cone observation: "
      f"{sum(bool(o.visible.any()) for o in obs_history)} / {N_STEPS}")

# Sanity check: rotation is happening (IC spin from FixedAttitude sampler).
quat_t0 = state_history[0].guards.quat[0]
quat_tN = state_history[-1].guards.quat[0]
dot = float(jnp.clip(jnp.abs(quat_t0 @ quat_tN), 0.0, 1.0))
total_angle_deg = float(jnp.rad2deg(2.0 * jnp.arccos(dot)))
print(f"Guard quaternion rotation t=0→t=N: {total_angle_deg:.1f} deg")
print(f"  (passive rotation: 0.1 deg/s/axis × 10s/step × 200 steps = ~346 deg net Euler-axis rotation; quaternion angle wraps at 180 deg)")

# ── Build a Trajectory pytree from the loop history for RolloutScene ──────
traj_heuristic = Trajectory(
    env_state=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *state_history),
    sides=BySide(
        guard=SideTrajectory(
            obs=jnp.stack(guard_obs_flat_history, axis=0),
            action=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *guard_action_history),
            reward=jnp.stack(reward_history, axis=0),
            done=jnp.stack(episode_done_history, axis=0),
            policy_state=None,
        ),
        bandit=SideTrajectory(
            obs=jnp.stack(bandit_obs_flat_history, axis=0),
            action=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *bandit_action_history),
            reward=jnp.zeros(N_STEPS, dtype=jnp.float32),
            done=jnp.stack(episode_done_history, axis=0),
            policy_state=None,
        ),
    ),
    episode_done=jnp.stack(episode_done_history, axis=0),
    controlled_side=Side.GUARD,
)

# Stack belief histories for RolloutScene.
belief_history_stacked = BySide(
    guard=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *belief_guard_history),
    bandit=jax.tree_util.tree_map(lambda *xs: jnp.stack(xs, axis=0), *belief_bandit_history),
)

# Sanity: check belief covariance Frobenius norms at final step.
g_cov_norm = float(jnp.linalg.norm(belief_history_stacked.guard.cov[-1, 0, 1], ord="fro"))
b_cov_norm = float(jnp.linalg.norm(belief_history_stacked.bandit.cov[-1, 0, 1], ord="fro"))
print(f"Final guard belief cov norm (bandit block):  {g_cov_norm:.4f}")
print(f"Final bandit belief cov norm (guard block):  {b_cov_norm:.4f}")

Episode return: 0.00
Steps with at least one in-cone observation: 16 / 200
Guard quaternion rotation t=0→t=N: 15.3 deg
  (passive rotation: 0.1 deg/s/axis × 10s/step × 200 steps = ~346 deg net Euler-axis rotation; quaternion angle wraps at 180 deg)


Final guard belief cov norm (bandit block):  4.4260
Final bandit belief cov norm (guard block):  4.4260


## 5. RTN trajectory plots — all four projections

Four projections of the relative motion: R-T (in-plane), R-N (radial/cross-
track), T-N (along-track/cross-track), and the full 3D RTN view.


In [7]:
guard_rtns  = jnp.stack([s.guards.rtn  for s in state_history], axis=0)  # (T, 1, 6)
bandit_rtns = jnp.stack([s.bandits.rtn for s in state_history], axis=0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

plot_rt(guard_rtns,  ax=axes[0], label="guard")
plot_rt(bandit_rtns, ax=axes[0], label="bandit")
axes[0].legend(); axes[0].set_title("R-T plane")

plot_rn(guard_rtns,  ax=axes[1], label="guard")
plot_rn(bandit_rtns, ax=axes[1], label="bandit")
axes[1].legend(); axes[1].set_title("R-N plane")

plot_tn(guard_rtns,  ax=axes[2], label="guard")
plot_tn(bandit_rtns, ax=axes[2], label="bandit")
axes[2].legend(); axes[2].set_title("T-N plane")

fig.tight_layout()
plt.savefig("/tmp/rtn_projections.png", dpi=80)
plt.show()

fig3d = plt.figure(figsize=(7, 6))
ax3d = fig3d.add_subplot(111, projection="3d")
plot_rtn_3d(guard_rtns,  ax=ax3d, label="guard")
plot_rtn_3d(bandit_rtns, ax=ax3d, label="bandit")
ax3d.legend()
ax3d.set_title("3D RTN trajectories")
plt.tight_layout()
plt.savefig("/tmp/rtn_3d.png", dpi=80)
plt.show()


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_69317/2638689214.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_69317/2638689214.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Visibility heatmap — in-cone observation timing (3D)

Same heatmap as in the 2D notebook.  In 3D the bandit can drift in cross-
track (N) as well as in-plane, so the visibility pattern may differ from
the 2D case even with the same boresight configuration.


In [8]:
visibility = jnp.stack([o.visible for o in obs_history], axis=0)  # (T, 1, 2)
vis_guard = visibility[:, 0, :]  # (T, 2)

fig, ax = plt.subplots(figsize=(10, 3))
ax.imshow(vis_guard.T, aspect="auto", origin="lower",
          cmap="Greys", interpolation="nearest",
          vmin=0, vmax=1)
ax.set_xlabel("step")
ax.set_ylabel("entity index")
ax.set_yticks([0, 1])
ax.set_yticklabels(["guard (self)", "bandit"])
ax.set_title("In-cone visibility over time (3D guard view)  — white=visible, black=not")
fig.tight_layout()
plt.savefig("/tmp/visibility_heatmap_3d.png", dpi=80)
plt.show()

n_visible = int(vis_guard[:, 1].sum())
print(f"Steps bandit was in-cone: {n_visible} / {N_STEPS}  ({100*n_visible/N_STEPS:.1f}%)")


Steps bandit was in-cone: 16 / 200  (8.0%)


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_69317/1315594087.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Reward curve


In [9]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(rewards_arr[:, 0] if rewards_arr.ndim > 1 else rewards_arr, lw=1.5, color="steelblue")
mean_r = float(rewards_arr.mean())
ax.axhline(mean_r, color="red", ls="--", lw=1, label=f"mean={mean_r:.2f}")
ax.set_xlabel("step")
ax.set_ylabel("reward")
ax.set_title("Guard reward per step (3D RTN)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/reward_curve_3d.png", dpi=80)
plt.show()


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_69317/1128907772.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Propellant mass


In [10]:
prop_mass = jnp.stack([s.guards.propellant_mass for s in state_history], axis=0)  # (T, 1)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(prop_mass[:, 0], color="darkorange")
ax.set_xlabel("step")
ax.set_ylabel("propellant mass (kg)")
ax.set_title("Guard propellant over time (flat = no translational maneuvers)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/propellant_mass_3d.png", dpi=80)
plt.show()


/var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/ipykernel_69317/472794504.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Persistence — save and reload via HDF5


In [11]:
from orbital_game.rollout import rollout as _rollout

zero_g = ZeroControl(n_vehicles=cfg.n_guards, command_cls=env.guard_command_cls)
zero_b = ZeroControl(n_vehicles=cfg.n_bandits, command_cls=env.bandit_command_cls)

traj_save = _rollout(
    env,
    BySide(guard=zero_g, bandit=zero_b),
    BySide(guard=lambda c, s, k: None, bandit=lambda c, s, k: None),
    jax.random.PRNGKey(99),
    n_steps=cfg.max_steps,
)

tmp = pathlib.Path(tempfile.mkdtemp())
out = tmp / "rollout_attitude_3d.h5"
save_run(out, cfg, traj_save)

loaded_cfg, loaded_traj = load_run(out)
print(f"file written:           {out}  ({out.stat().st_size:,} bytes)")
print(f"loaded config matches:  {loaded_cfg.n_guards == cfg.n_guards and loaded_cfg.dt == cfg.dt}")
attitude_keys = [k for k in sorted(loaded_traj.keys()) if "quat" in k or "omega" in k or "torque" in k]
print(f"attitude-related keys:  {attitude_keys}")
print(f"reward round-trips:     {bool(jnp.allclose(loaded_traj['sides.guard.reward'], traj_save.sides.guard.reward))}")


file written:           /var/folders/66/25bqc2bn19bb84ddx70zwv9m0000gn/T/tmp2j7p9xj5/rollout_attitude_3d.h5  (122,820 bytes)
loaded config matches:  True
attitude-related keys:  ['env_state.bandits.applied_torque', 'env_state.bandits.omega', 'env_state.bandits.quat', 'env_state.guards.applied_torque', 'env_state.guards.omega', 'env_state.guards.quat', 'sides.guard.action.torque']
reward round-trips:     True


## 10. Animation — 3D cones sweeping with attitude

In 3D mode `RolloutScene` uses `draw_cone_3d` to render each sensor cone
as a 3D surface rotated into world frame by the vehicle's quaternion.
The cone sweeps visibly as the guard executes the pointing heuristic.

`cone_length_m` defaults to `None`, which renders cones to 2.5× the axis limit —
well past the plot edge, making the unbounded nature of the conical sensor visually clear.


Save the animation in two formats: MP4 (smaller, requires ffmpeg) and GIF
(larger but universally embeddable in markdown / web previews / Slack /
GitHub READMEs without a player). The `save_animation` helper dispatches
on file extension — `.mp4` uses ffmpeg, `.gif` uses matplotlib's bundled
PillowWriter.

In [12]:
# Build scene from the heuristic rollout (traj_heuristic from the rollout
# cell above). The IC body rates come from FixedAttitude via attitude_sampler,
# so no manual omega injection is needed — the trajectory already has the
# correct initial spin baked in from env.reset().
scene = RolloutScene(
    traj=traj_heuristic,
    cfg=cfg,
    dt=cfg.dt,
    show_cubes=True,
    show_trail=True,
    show_sensor_cones=True,
    show_thrust=False,
    show_sensor_range=False,
    show_belief=True,
    belief_history=belief_history_stacked,
    title_prefix="Attitude 3D RTN  ",
)
print(f"resolved mode:  {scene.resolved_mode}")
print(f"axis limit (m): {scene.axis_limit_m:.0f}")
print(f"n_frames:       {scene.n_frames}")
print(f"eff cone length (m): {2.5 * scene.axis_limit_m:.0f}  (2.5 × axis_limit)")

import shutil
out_dir = pathlib.Path("../outputs")
out_dir.mkdir(parents=True, exist_ok=True)

# GIF always works (PillowWriter is bundled with matplotlib)
gif_path = out_dir / "workflow_3d_rtn_attitude.gif"
save_animation(scene, str(gif_path), fps=15, dpi=90)
print(f"saved {gif_path}  ({gif_path.stat().st_size:,} bytes)")

# MP4 is smaller and smoother but needs ffmpeg on PATH
if shutil.which("ffmpeg") is not None:
    mp4_path = out_dir / "workflow_3d_rtn_attitude.mp4"
    save_animation(scene, str(mp4_path), fps=15, dpi=90)
    print(f"saved {mp4_path}  ({mp4_path.stat().st_size:,} bytes)")
else:
    print("ffmpeg not available — skipping MP4 export")

# In-notebook playback: prefer MP4 (smaller, smoother) when ffmpeg produced
# one, fall back to the GIF otherwise. ``embed=False`` keeps the notebook
# small by referencing the file path instead of base64-encoding the bytes.
from IPython.display import Image, Video, display

if shutil.which("ffmpeg") is not None and mp4_path.exists():
    display(Video(str(mp4_path), embed=False))
else:
    display(Image(str(gif_path)))


resolved mode:  3d
axis limit (m): 2224
n_frames:       200
eff cone length (m): 5561  (2.5 × axis_limit)


saved ../outputs/workflow_3d_rtn_attitude.gif  (14,823,875 bytes)


saved ../outputs/workflow_3d_rtn_attitude.mp4  (775,794 bytes)
